# LIPID MAPS — Comprehensive Lipid Structure Database

**LIPID MAPS** is an [ELIXIR Core Data Resource](https://elixir-europe.org/platforms/data/core-data-resources) and the world's most comprehensive lipid classification system and structure database. It hosts the **LIPID MAPS Structure Database (LMSD)** containing ~47,000 unique, experimentally characterised lipid structures, along with the **LIPID MAPS Proteome Database (LMPD)** for lipid-related proteins.

Lipids are a structurally diverse class of biomolecules defined by their hydrophobicity. They serve as membrane building blocks, signalling molecules, and energy stores. The LIPID MAPS classification system organises them into **eight main categories** based on their biochemical and structural properties:

| Abbreviation | Full Name | Example |
|---|---|---|
| **FA** | Fatty Acyls | Palmitic acid (C16:0) |
| **GL** | Glycerolipids | Triacylglycerol (TAG) |
| **GP** | Glycerophospholipids | Phosphatidylcholine (PC) |
| **SP** | Sphingolipids | Ceramide |
| **ST** | Sterol Lipids | Cholesterol |
| **PR** | Prenol Lipids | Coenzyme Q10 |
| **SL** | Saccharolipids | Lipid A |
| **PK** | Polyketides | Aflatoxin B1 |

**References:**
- Fahy, E. et al. (2009). Update of the LIPID MAPS comprehensive classification system for lipids. *J. Lipid Res.* 50, S9–S14. https://doi.org/10.1194/jlr.R800095-JLR200
- LIPID MAPS website: https://www.lipidmaps.org/
- REST API: https://www.lipidmaps.org/rest/

In [1]:
import requests
import time
import re
import json
from pathlib import Path

import polars as pl
import pandas as pd

## TODO

### Ingest data
- [x] Connect to LIPID MAPS REST API
- [x] Download lipid structure records by category (FA, GL, GP, SP, ST, PR, SL, PK)
- [x] Parse records into a Polars DataFrame with structural metadata

### Explore and clean
- [ ] Category distributions — bar chart of record counts per category / sub-class
- [ ] Formula / mass distributions — histogram of exact masses
- [ ] Missing structural data — identify records lacking InChIKey, formula, or mass

### Chemical space analysis
- [ ] Molecular weight distributions per category (overlapping KDEs)
- [ ] Degree of unsaturation (DoU = (2C + 2 + N − H) / 2) distribution
- [ ] Carbon chain length analysis for acyl-containing lipids

### Functional annotation
- [ ] Biological roles and disease associations from LMSD metadata
- [ ] Metabolic pathway mapping using KEGG/Reactome cross-references

### Visualization
- [ ] Chemical space scatter: MW vs. degree of unsaturation, coloured by category
- [ ] Category treemap: category → main class → sub-class → count
- [ ] Structure count per sub-class horizontal bar chart

### Statistical analysis
- [ ] Mass distribution fitting (normal / log-normal / mixture models) per category
- [ ] Structural diversity metrics (Tanimoto-based diversity) per category

## 1. Ingest Data

### 1.1 Connect to LIPID MAPS API

In [ ]:
LIPIDMAPS_BASE = "https://www.lipidmaps.org/rest"


def lm_get(endpoint: str, params: dict | None = None) -> dict | list:
    """Send a GET request to the LIPID MAPS REST API.

    Parameters
    ----------
    endpoint : str
        URL path appended to the base URL, e.g. ``/compound/lm_id/LMFA01010001/details/json``.
    params : dict or None, optional
        Query-string parameters forwarded to ``requests.get``.

    Returns
    -------
    dict or list
        Parsed JSON response body.

    Raises
    ------
    requests.HTTPError
        If the server returns a 4xx or 5xx status code.
    """
    url = f"{LIPIDMAPS_BASE}{endpoint}"
    # Be polite to the public API — 1 second between requests
    time.sleep(1)
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()


# --- Connectivity check ---
# Fetch a known fatty acid (palmitic acid, LMFA01010001) to verify the API is reachable
# Correct endpoint format: /compound/lm_id/{LM_ID}/details/json
test_record = lm_get("/compound/lm_id/LMFA01010001/details/json")
print("API connectivity OK — fetched:", test_record.get("name", test_record))

### 1.2 Download Lipid Structure Records

In [ ]:
# Create the data directory next to this notebook
DATA_DIR = Path("/Users/alice/github/elixir-of-life/lipid-maps/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# All eight top-level LIPID MAPS category codes
CATEGORIES = ["FA", "GL", "GP", "SP", "ST", "PR", "SL", "PK"]

for code in CATEGORIES:
    out_path = DATA_DIR / f"lmsd_{code}.json"

    # Skip download if the file already exists (idempotent ingest)
    if out_path.exists():
        with out_path.open() as fh:
            records = json.load(fh)
        print(f"{code}: {len(records):>6,} records  (cached)")
        continue

    # Fetch all records for this category from the LMSD REST endpoint
    records = lm_get(f"/compound/category/{code}/all/json")

    # Persist raw JSON so subsequent runs never re-hit the API
    with out_path.open("w") as fh:
        json.dump(records, fh)

    print(f"{code}: {len(records):>6,} records  (downloaded)")

### 1.3 Parse into DataFrame

In [ ]:
# Columns we want to keep from the raw JSON records
KEEP_COLS = [
    "lm_id",
    "name",
    "systematic_name",
    "category",
    "main_class",
    "sub_class",
    "exact_mass",
    "formula",
    "inchi_key",
    "pubchem_cid",
]


def _snake(s: str) -> str:
    """Convert a CamelCase or mixed-case string to snake_case."""
    # Insert underscore before upper-case letters preceded by lower-case letters
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)
    return s.lower()


rows: list[dict] = []

for code in CATEGORIES:
    raw: list[dict] = json.loads((DATA_DIR / f"lmsd_{code}.json").read_text())
    for rec in raw:
        # Extract only the desired fields; missing fields default to None
        row = {col: rec.get(col) for col in KEEP_COLS}
        rows.append(row)

# Build Polars DataFrame; normalise column names to snake_case for consistency
df = pl.DataFrame(rows).rename({c: _snake(c) for c in KEEP_COLS})

# Cast exact_mass to float (arrives as a string from the API)
df = df.with_columns(pl.col("exact_mass").cast(pl.Float64, strict=False))

print(f"DataFrame shape: {df.shape}")
df.head(5)

## DataFrame Column Reference

| Column | Type | Description |
|---|---|---|
| `lm_id` | str | **LIPID MAPS unique identifier.** Format: `LM` + 2-letter category code + 2-digit main-class + 2-digit sub-class + 6-digit serial, e.g. `LMFA01010001`. The first two letters after `LM` encode the category (FA = Fatty Acyls, GP = Glycerophospholipids, etc.), making the ID self-describing. |
| `name` | str | Common or shorthand name (e.g. "Palmitic acid", "PC 16:0/18:1"). |
| `systematic_name` | str | IUPAC systematic name, if available. Often absent for complex lipids. |
| `category` | str | Top-level LIPID MAPS category full name (e.g. "Fatty Acyls"). |
| `main_class` | str | Second hierarchy level (e.g. "Fatty Acids and Conjugates"). |
| `sub_class` | str | Third hierarchy level (e.g. "Straight chain fatty acids"). |
| `exact_mass` | float64 | **Monoisotopic exact mass (Da).** Calculated from the molecular formula using the most abundant isotope of each element (¹H, ¹²C, ¹⁴N, ¹⁶O, etc.). This is the mass measured by high-resolution mass spectrometry and differs from the average atomic mass reported on chemical data sheets. |
| `formula` | str | Molecular formula, e.g. `C16H32O2`. |
| `inchi_key` | str | 27-character hashed InChI key — a compact, fixed-length identifier for chemical structures that enables exact-structure database lookups and deduplication. |
| `pubchem_cid` | str | PubChem Compound ID, enabling cross-referencing to PubChem for additional properties, bioactivity data, and literature links. |